# glean in action

`glean` is the *what it means* verb: text in, one ranked list of entries out. This notebook builds a small
namespace from this repository's own files, indexes it, and walks through what a glean result carries —
the ranked rows, the chunk regions with their line bounds, the query-biased previews, scoping, the
freshness overlay, and the term statistics the ranking used.

Today glean is lexical-only (BM25 over block postings in the Rust engine). The vector leg and fusion are
the next specs; the verb, the result shape, and the renderer stay the same when they land.

Run the cells top to bottom. Everything is local: a throwaway sqlite file in a temp directory.

In [1]:
import tempfile
from pathlib import Path as FsPath

from vfs import VirtualFileSystem
from vfs.models import Entry
from vfs.paths import Path
from vfs.results.preview import select_preview
from vfs.storage.backends import DatabaseStorage

# Find the repository root from wherever this notebook is opened.
root = FsPath.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
workdir = FsPath(tempfile.mkdtemp())
print("repo:", root)
print("scratch:", workdir)

repo: /Users/claygendron/Git/Repos/vfs
scratch: /var/folders/_c/n40__q_16zs0cbn45nb3c_nw0000gn/T/tmprn74vahr


## 1. Build a namespace

Files become `Entry` rows. We take the library's own source and the research memos — code and prose in one corpus,
about 200 files. `parents=True` creates the directories along the way.

In [2]:
def entries_under(folder: str, pattern: str) -> list[Entry]:
    out = []
    for file in sorted((root / folder).rglob(pattern)):
        if "__pycache__" in file.parts:
            continue
        out.append(Entry(path=Path("/" + file.relative_to(root).as_posix()), content=file.read_text()))
    return out

corpus = entries_under("src/vfs", "*.py") + entries_under("context/research", "*.md")
storage = DatabaseStorage(url=f"sqlite+aiosqlite:///{workdir}/glean.sqlite")
written = await storage.write(entries=corpus, parents=True)
print(written.success, len(corpus), "files written")

True 232 files written


## 2. Index it

`reindex()` splits each file into chunks, builds the gram index grep uses, and builds the lexical index glean uses.
Until it runs, glean still answers — from the *overlay*, the live text of unindexed files (section 6).

In [3]:
indexed = await storage.reindex()
print(indexed.success)
print(indexed.to_str())

True



## 3. The first glean

The rendered text is what an agent reads. Rows come back **in rank order** — never sorted by path. Each entry prints a
header with its score, then each of its best chunk regions as a `path:start-end` locator with a short preview
quoted beneath it. Query terms are bolded in Markdown.

In [4]:
result = await storage.glean(query="freshness overlay two-read protocol", limit=5)
print(result.to_str())

/src/vfs/storage/backends/database/glean.py  score=1.0000
/src/vfs/storage/backends/database/glean.py:1-34
> query time and scored by the same engine with the epoch's statistics,
> so the **two** sides share one formula, one accumulation order and one
> scale. The pointer **read** is advisory and re-issued after the fetch (the
> **two**-**read** **protocol**); a moved pointer raises :class:`StaleSnapshot` for
/src/vfs/storage/backends/database/glean.py:454-495
>     epoch's corpus size plus the **overlay**'s); with no epoch at all the
>     **overlay** is the corpus, mean length included. The scored entries'
>     bodies ride back beside the hits: their previews are cut from the
>     text al**read**y fetched here, never from a second **read**.
/src/vfs/storage/backends/database/glean.py:199-225
>     truncations: list[tuple[str, dict[str, Any] | None]] = []
>
>     epoch, **overlay**_empty = await pointer_with_**overlay**(session, tables)
>     await seam("glean:after-pointer-**read**

## 4. What a row carries

The same result as data. One `Observation` per entry; `score` is the entry's fused score (min-max scaled, so the best
hit is `1.0`); `matches` holds up to three chunk regions, best first, each with absolute 1-indexed line bounds, the raw
chunk text in `content`, and the preview with its own bounds — a sub-range of the region.

In [6]:
for row in result.observations[:3]:
    print(f"{row.path}  score={row.score:.4f}")
    for region in row.matches or []:
        print(f"  lines {region.start}-{region.end}  chunk score {region.score:.4f}")
        print(f"  preview lines {region.preview_start}-{region.preview_end}: {region.preview!r}")
        print(f"  raw text: {len(region.content or '')} chars")
    print()

/src/vfs/storage/backends/database/glean.py  score=1.0000
  lines 1-34  chunk score 1.0000
  preview lines 29-32: "query time and scored by the same engine with the epoch's statistics,\nso the **two** sides share one formula, one accumulation order and one\nscale. The pointer **read** is advisory and re-issued after the fetch (the\n**two**-**read** **protocol**); a moved pointer raises :class:`StaleSnapshot` for"
  raw text: 1963 chars
  lines 454-495  chunk score 0.5705
  preview lines 474-477: "    epoch's corpus size plus the **overlay**'s); with no epoch at all the\n    **overlay** is the corpus, mean length included. The scored entries'\n    bodies ride back beside the hits: their previews are cut from the\n    text al**read**y fetched here, never from a second **read**."
  raw text: 1982 chars
  lines 199-225  chunk score 0.5240
  preview lines 215-218: '    truncations: list[tuple[str, dict[str, Any] | None]] = []\n\n    epoch, **overlay**_empty = await pointer_with_**overlay**(

## 5. Projections

The renderer takes a projection — which columns to show. The native glean vocabulary (`path`, `score`, `matches`)
prints as ranked lines; ask for a row-level field and it switches to a Markdown table, still in rank order.

In [7]:
print(result.to_str(projection=("path", "score")))
print()
print(result.to_str(projection=("path", "score", "size_bytes")))

/src/vfs/storage/backends/database/glean.py  score=1.0000
/context/research/2026-08-10-mcp-2026-07-28-stateless-revision.md  score=0.8759
/context/research/2026-08-25-chunking-arc-landing-review.md  score=0.6851
/context/research/studies/2026-07-25-multimodal-storage/vfs-search-ground-truth.md  score=0.6736
/context/research/studies/2026-08-17-search-storage-organizations/notes-gin-pgtrgm.md  score=0.6528

| path                                                                                  |  score | size_bytes |
| ------------------------------------------------------------------------------------- | -----: | ---------: |
| /src/vfs/storage/backends/database/glean.py                                           | 1.0000 |      26045 |
| /context/research/2026-08-10-mcp-2026-07-28-stateless-revision.md                     | 0.8759 |      28455 |
| /context/research/2026-08-25-chunking-arc-landing-review.md                           | 0.6851 |      16318 |
| /context/research/studies/20

## 6. Scope

Scoping is grep's: `globs`, `globs_not`, `ext`, `ext_not`, and `paths`. Whichever rung the ladder takes internally
(candidate ids for a narrow scope, score-then-gate for a wide one), a scoped call never returns an out-of-scope entry.

In [8]:
only_docs = await storage.glean(query="block postings per term", limit=5, globs=("context/**",))
print(only_docs.to_str(projection=("path", "score")))
print()
only_code = await storage.glean(query="block postings per term", limit=5, ext=("py",))
print(only_code.to_str(projection=("path", "score")))

/context/research/studies/2026-08-26-bm25-storage/vectorchord-bestmatch-lucene.md  score=1.0000
/context/research/studies/2026-08-26-bm25-storage/paradedb-tantivy.md  score=0.9317
/context/research/studies/2026-08-26-bm25-storage/pg_textsearch.md  score=0.9072
/context/research/studies/2026-08-26-bm25-storage/prototype-benchmark.md  score=0.8948
/context/research/studies/2026-08-26-bm25-storage/review-validation.md  score=0.8192

/src/vfs/models/lexical.py  score=1.0000
/src/vfs/storage/backends/database/glean.py  score=0.9656
/src/vfs/models/rows.py  score=0.8834
/src/vfs/storage/backends/database/lexical.py  score=0.6966
/src/vfs/storage/backends/database/pathterms.py  score=0.5026


## 7. The freshness overlay

A file written after the last reindex is not in the lexical index yet. Glean still finds it: unindexed entries are
scored from their live text with the index's own statistics, so the two sides share one scale. An overlay hit answers
as one whole document — its region spans the file, carries no chunk text, and its preview is cut from the body that was
already read to score it (no second fetch).

In [9]:
fresh = Entry(
    path=Path("/notes/lantern.md"),
    content="# Lantern protocol\n\nA lantern is a signal the router raises when a mount goes dark.\n"
    "Lanterns are cheap; the light is the point.\n",
)
await storage.write(entries=[fresh], parents=True)

before = await storage.glean(query="lantern signal", limit=3)
print("before reindex:")
print(before.to_str())
hit = before.observations[0]  # the fresh file outranks the memos that mention lanterns
[region] = hit.matches or []
print("\nregion bounds:", (region.start, region.end), " content carried:", region.content is not None)

await storage.reindex()
after = await storage.glean(query="lantern signal", limit=3)
print("\nafter reindex:")
print(after.to_str())

before reindex:
/notes/lantern.md  score=1.0000
/notes/lantern.md:1-4
> # **Lantern** protocol
>
> A **lantern** is a **signal** the router raises when a mount goes dark.
> **Lantern**s are cheap; the light is the point.
/context/research/studies/2026-08-26-glean/engine-matrix.md  score=0.3508
/context/research/studies/2026-08-26-glean/engine-matrix.md:317-336
>   k3=8.0` and Robertson–Sparck Jones weights (docs, formula quoted on the
>   RANK page); `CONTAINSTABLE` ranks by `HitCount * 16 * log2((2 +
>   IndexedRowCount)/KeyRowCount) / MaxOccurrence` with `MaxOccurrence`
>   bucketed into 32 ranges (docs). `CONTAINSTABLE(..., '**lantern**')` returned
/context/research/studies/2026-08-26-glean/engine-matrix.md:30-64
> max over chunks, `LIMIT n` — runs as **one statement** on Postgres, MariaDB,
> SQL Server, Oracle and SQLite, and returns the **same top-5, same scores**
> on all five (**VERIFIED**; corpus: 100 entries × 4 chunks, 8-dim synthetic
> vectors, lexical target "**lantern**" p

## 8. The statistics behind the ranking

Every glean result carries a `lexical_stats` extra: the corpus size, the mean document length, and each query term's
document frequency and idf. A term missing here occurred nowhere in the corpus. Note that identifiers split —
`line_start` becomes `line_start`, `line`, and `start` — which is also why the bolder marks short terms inside longer words.

In [10]:
stats = (await storage.glean(query="chunk line_start embedding", limit=1)).lexical_stats
print("documents:", stats["n_docs"], " mean length:", round(stats["avg_dl"], 1))
for term, facts in stats["terms"].items():
    print(f"  {term:12s} df={facts['df']:4d}  idf={facts['idf']:.3f}")

documents: 2687  mean length: 229.4
  chunk        df= 528  idf=1.627
  line_start   df=  25  idf=4.658
  line         df= 340  idf=2.066
  start        df= 150  idf=2.883
  embedding    df= 178  idf=2.712


## 9. The preview selector on its own

`select_preview` is a pure function: chunk text, the chunk's first line number, the query's folded terms. It folds the
chunk once, scores each line by the distinct terms it holds (whole-word hits count double a substring hit), slides a
4-line window, bolds merged spans, and trims to 160 characters a line and 480 a preview. No term in the chunk → the head
of the chunk, unbolded. The display budgets are keyword arguments, never glean parameters.

In [11]:
chunk = (root / "src/vfs/results/preview.py").read_text()
excerpt = select_preview(chunk, 1, ["fold", "window", "cap"])
print(f"lines {excerpt.start}-{excerpt.end}, matched={excerpt.matched}")
print(excerpt.text)
print()
narrow = select_preview(chunk, 1, ["fold", "window", "cap"], window=2, line_cap=60)
print(f"lines {narrow.start}-{narrow.end}")
print(narrow.text)
print()
head = select_preview(chunk, 1, ["zzqx"])
print(f"no hit -> head window, lines {head.start}-{head.end}, matched={head.matched}")

lines 97-100, matched=True
    **cap**ped bonus per extra occurrence and a bonus for adjacent distinct
    terms. A **window** scores the sum of its lines plus a bonus per distinct
    term it covers; the best **window** wins, the earliest on ties. With no
    term anywhere in the chunk, the head **window** comes back unbolded.

lines 4-5
text alone: one **fold** of the chunk, a score per line from the…
terms, the best short **window** of lines, the terms bolded in M…

no hit -> head window, lines 1-4, matched=False


## 10. Through the router, across mounts

A `VirtualFileSystem` routes one namespace over many stores. Mount two backends and glean from the root: each mount
answers in its own coordinates and the router rebases the paths on the way back. (The cross-mount *rerank* is a later spec; today each mount's scores are on its own scale, so two mounts can both
show a `1.0000`.)

In [12]:
code_store = DatabaseStorage(url=f"sqlite+aiosqlite:///{workdir}/code.sqlite")
docs_store = DatabaseStorage(url=f"sqlite+aiosqlite:///{workdir}/docs.sqlite")
await code_store.write(entries=entries_under("src/vfs", "*.py"), parents=True)
await docs_store.write(entries=entries_under("context/research", "*.md"), parents=True)
await code_store.reindex()
await docs_store.reindex()

fs = VirtualFileSystem()
await fs.add_mount(code_store, "/code")
await fs.add_mount(docs_store, "/docs")

merged = await fs.glean("two-read protocol epoch pointer", limit=6)
print(merged.to_str(projection=("path", "score")))
print()
scoped = await fs.glean("two-read protocol epoch pointer", limit=3, paths=("/docs",))
print(scoped.to_str(projection=("path", "score")))

/code/src/vfs/storage/backends/database/glean.py  score=1.0000
/docs/context/research/2026-08-13-glob-grep-indexing-review-campaign.md  score=1.0000
/docs/context/research/studies/2026-08-26-glean/centrality-and-read-signals.md  score=0.8563
/docs/context/research/2026-07-13-database-storage-grep-index.md  score=0.7428
/code/src/vfs/storage/backends/database/indexing.py  score=0.7425
/code/src/vfs/storage/backends/database/reads.py  score=0.7212

/docs/context/research/2026-08-13-glob-grep-indexing-review-campaign.md  score=1.0000
/docs/context/research/studies/2026-08-26-glean/centrality-and-read-signals.md  score=0.8563
/docs/context/research/2026-07-13-database-storage-grep-index.md  score=0.7428


## 11. Clean up

In [13]:
await fs.close()
await storage.close()
print("closed")

closed
